# Analyse de la base du bloc opératoire (2019-2022)

Ce notebook **nettoie, analyse et prépare** la base `donees_bloc_anonyme_pour_centrale_2026.xlsx` pour toute la suite du projet (recuit simulé, machine learning).

Il s'appuie sur le module `preparation_donnees.py` (à placer dans le même dossier) qui :
- renomme et type proprement les 30 colonnes du fichier, et crée ~50 variables utiles ;
- calcule la **durée de séjour à partir des dates d'entrée et de sortie** — la colonne « durée de séjour » du fichier est fausse pour environ un séjour sur six ;
- indique pour chaque variable **à quel moment elle est connue** (consultation, anesthésie, après l'opération, après la sortie), pour éviter les fuites d'information en machine learning ;
- écrit le dossier `base_preparee/` : base propre (`.pkl`, `.csv`, `.xlsx`), dictionnaire des variables, rapport de qualité, occupation journalière des lits, profils des interventions et des chirurgiens.

**Plan** : 1. Qualité des données · 2. Activité dans le temps · 3. Durées de séjour · 4. Les actes chirurgicaux (CCAM) · **5. Ce qui détermine la durée d'hospitalisation (diagnostic, acte, âge)** · **6. Prédire la durée d'hospitalisation d'un nouveau patient** · **7. Classification des urgences** · 8. Patients · 9. Bloc opératoire · 10. Occupation des lits · 11. Profils · 12. Corrélations · 13. Base prête pour le machine learning · 14. Utiliser la base dans les autres codes

Trois angles d'analyse se complètent : le **libellé du type d'intervention** (saisi par le service, 1 433 libellés), le **diagnostic** (CIM-10, pourquoi on opère) et l'**acte CCAM** (ce qu'on fait, codé selon la nomenclature nationale, section 4).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from preparation_donnees import (charger_brut, preparer, exporter, charger_base_preparee, rapport_qualite, dictionnaire, JOURS,
                                 occupation_journaliere, profil_interventions, profil_chirurgiens, profil_actes, jeu_ml,
                                 decoupage_temporel, variables_connues, CIBLES, COVID, VACANCES_ZONE_B)

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110})
pd.set_option("display.max_columns", 40, "display.width", 200)

brut = charger_brut()
df = preparer(brut)
exporter(brut, df)                    # écrit base_preparee/ (à refaire seulement si la base change)
print(f"Base brute : {brut.shape[0]} lignes x {brut.shape[1]} colonnes")
print(f"Base préparée : {df.shape[0]} lignes x {df.shape[1]} colonnes")
df.head()

## 1. Qualité des données

### 1.1 Contrôles automatiques

In [ ]:
display(rapport_qualite(brut, df))

### 1.2 La durée de séjour du fichier est fausse : on utilise les dates

Comparaison de la colonne du fichier (« Durée séjour, 1 pour ambu ») et de la durée calculée par les dates (`nuits_total + 1`). Sur la diagonale, les deux sont d'accord ; ailleurs, la colonne du fichier est fausse.

In [ ]:
t = pd.crosstab(df.duree_sejour_declaree.clip(upper=8).rename("déclarée (8 = 8 et +)"),
                df.duree_sejour_jours.clip(upper=8).rename("calculée par les dates (8 = 8 et +)"))
plt.figure(figsize=(8, 5.5))
sns.heatmap(t, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Durée de séjour (jours) : colonne du fichier contre dates")
plt.show()
inc = df[df.duree_declaree_incoherente]
display(Markdown(f"**{len(inc)} séjours ({len(inc) / len(df):.1%})** ont une durée déclarée fausse, dont "
                 f"**{((inc.duree_sejour_declaree == 1) & (inc.nuits_total > 0)).sum()}** déclarés ambulatoires alors que "
                 f"le patient a dormi au moins une nuit. Types d'intervention les plus touchés :"))
display(inc.type_intervention_norm.value_counts().head(10).to_frame("séjours incohérents"))

### 1.3 Valeurs manquantes
Comme dans le cours (`isna().sum()`), on regarde les variables incomplètes. Les heures à 00:00 ont été considérées comme inconnues ; les diagnostics associés et les actes secondaires manquent naturellement (pas de comorbidité, un seul acte).

In [ ]:
manq = df.isna().sum()
manq = manq[manq > 0].sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=manq.values / len(df) * 100, y=manq.index, color="C0")
plt.xlabel("% de valeurs manquantes"); plt.title("Variables incomplètes")
plt.show()

## 2. L'activité dans le temps

In [ ]:
sem = df.set_index("date_inter").resample("W-MON").size()
fig, ax = plt.subplots(figsize=(14, 4))
for a, b in COVID:
    ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color="red", alpha=0.12)
ax.plot(sem.index, sem.values, lw=1)
ax.plot(sem.index, sem.rolling(8, center=True).mean(), lw=2, color="k", label="moyenne sur 8 semaines")
ax.set_ylabel("interventions par semaine"); ax.set_title("Activité hebdomadaire (rouge : confinements)"); ax.legend()
plt.show()

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="annee", ax=axs[0], color="C0"); axs[0].set_title("Interventions par année")
sns.countplot(data=df, x="jour_semaine_nom", ax=axs[1], color="C0"); axs[1].set_title("Par jour de la semaine"); axs[1].tick_params(axis="x", rotation=30)
sns.countplot(data=df, x="mois", ax=axs[2], color="C0"); axs[2].set_title("Par mois")
plt.tight_layout(); plt.show()
v = df.groupby("vacances_scolaires").size() / df.groupby("vacances_scolaires").date_inter.nunique()
display(Markdown(f"Interventions par jour opératoire : **{v.get(False, 0):.1f}** hors vacances scolaires, "
                 f"**{v.get(True, 0):.1f}** pendant les vacances."))

## 3. Les durées de séjour (calculées par les dates)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(df.nuits_total.clip(upper=15), bins=16, discrete=True, ax=axs[0])
axs[0].set_title("Nuits par séjour (15 = 15 et +)"); axs[0].set_xlabel("nuits")
sns.countplot(data=df, x="classe_sejour", ax=axs[1], color="C0"); axs[1].tick_params(axis="x", rotation=20)
axs[1].set_title("Classes de séjour")
df.groupby("annee").ambulatoire.mean().mul(100).plot.bar(ax=axs[2], rot=0)
axs[2].set_ylabel("% ambulatoire"); axs[2].set_title("Part d'ambulatoire par année")
plt.tight_layout(); plt.show()
hosp = df[~df.ambulatoire]
display(Markdown(f"- **{df.ambulatoire.mean():.0%}** d'ambulatoire ; parmi les hospitalisés, médiane **{hosp.nuits_total.median():.0f} nuits**, "
                 f"moyenne **{hosp.nuits_total.mean():.1f}** ; **{hosp.entree_veille.mean():.0%}** entrent la veille."))

In [ ]:
top = df.type_intervention_norm.value_counts().head(15).index
d = df[df.type_intervention_norm.isin(top)]
fig, axs = plt.subplots(1, 2, figsize=(17, 6))
ordre = d.groupby("type_intervention_norm").nuits_total.mean().sort_values().index
sns.boxplot(data=d, y="type_intervention_norm", x="nuits_total", order=ordre, ax=axs[0], showfliers=False)
axs[0].set_title("Nuits selon le type d'intervention (15 plus fréquents)"); axs[0].set_ylabel("")
(d.groupby("type_intervention_norm").ambulatoire.mean().reindex(ordre) * 100).plot.barh(ax=axs[1])
axs[1].set_xlabel("% ambulatoire"); axs[1].set_title("Part d'ambulatoire"); axs[1].set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ordre = df.groupby("chirurgien", observed=True).nuits_total.mean().sort_values().index
sns.violinplot(data=df[~df.ambulatoire], x="chirurgien", y="nuits_total", order=[o for o in ordre if o in set(df[~df.ambulatoire].chirurgien)],
               inner="quartile", cut=0, ax=ax)
ax.set_ylim(0, 15); ax.set_title("Nuits des patients hospitalisés selon le chirurgien")
plt.show()

**Effet du jour d'opération** : un patient opéré le jeudi ou le vendredi a plus de chances de rester le week-end. C'est ce qui relie le planning des vacations à l'occupation des lits.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
sns.barplot(data=df[~df.ambulatoire], x="jour_semaine_nom", y="nuits_total", ax=axs[0], errorbar=("ci", 95))
axs[0].set_title("Nuits moyennes des hospitalisés selon le jour d'opération"); axs[0].tick_params(axis="x", rotation=30)
sns.barplot(data=df, x="jour_semaine_nom", y="sejour_couvre_weekend", ax=axs[1], errorbar=None)
axs[1].set_title("Part des séjours qui couvrent une nuit de week-end"); axs[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 4. Les actes chirurgicaux (CCAM)

Chaque intervention porte jusqu'à 4 codes CCAM. Un code a 7 caractères :
- **2 lettres de topographie** — la 1re donne le grand appareil (N membre inférieur, M membre supérieur, L rachis, E vaisseaux, A système nerveux…), la 2e l'organe (NF genou, NE hanche, MJ épaule, EJ veines du membre inférieur…) ;
- **2 lettres d'action** — la 3e est l'action (Q = examen diagnostique, K = remplacement / prothèse…), la 4e la voie d'accès (C = endoscopique, c'est-à-dire sous arthroscopie) ;
- **3 chiffres**.

### 4.1 Le 1er acte de la liste n'est pas toujours l'intervention
Dans un quart des séjours, le 1er code est un **examen** (radiographie, examen de la conduction nerveuse…), pas l'opération. On travaille donc avec l'**acte chirurgical** : le 1er acte qui n'est ni un examen (3e lettre Q), ni une transfusion (F…), ni un supplément (Y…), ni un acte sans topographie (Z…).

In [ ]:
display(Markdown(f"- **{df.premier_acte_est_diagnostique.mean():.1%}** des séjours ont un examen en 1re position ; "
                 f"**{df.ccam_principal.nunique()}** codes en 1re position contre **{df.acte_chirurgical.nunique()}** actes chirurgicaux distincts.\n"
                 f"- Les 20 actes chirurgicaux les plus fréquents couvrent **{df.acte_chirurgical.value_counts().head(20).sum() / len(df):.0%}** "
                 f"des interventions, les 40 premiers **{df.acte_chirurgical.value_counts().head(40).sum() / len(df):.0%}**."))
exam = pd.concat([df[c] for c in ["ccam_1", "ccam_2", "ccam_3", "ccam_4"]]).dropna()
exam = exam[exam.str[2] == "Q"].value_counts().head(8)
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
sns.countplot(data=df, x="nb_actes_chirurgicaux", ax=axs[0], color="C0"); axs[0].set_title("Actes chirurgicaux par intervention")
sns.countplot(data=df, x="nb_actes_diagnostiques", ax=axs[1], color="C1"); axs[1].set_title("Examens associés par intervention")
df.acte_appareil.astype(str).value_counts().plot.barh(ax=axs[2]); axs[2].set_title("Grand appareil de l'acte chirurgical")
plt.tight_layout(); plt.show()
display(Markdown("**Examens les plus fréquents dans les codes** (3e lettre Q) :")); display(exam.to_frame("occurrences"))

### 4.2 Profil des actes
Pour chaque acte (au moins 20 interventions) : le libellé le plus fréquent dans la base et sa part (la cohérence entre code et libellé), le séjour, la consommation de lits, la durée en salle et sa variabilité (coefficient de variation = écart-type / moyenne).

In [ ]:
pa = profil_actes(df)
nom = lambda code: f"{code} — {pa.loc[code, 'libelle_le_plus_frequent'][:32]}" if code in pa.index else code
display(pa.head(30))

### 4.3 Quels actes remplissent les lits ?
C'est la question centrale pour lisser l'occupation : quelques actes concentrent l'essentiel des lits-nuits.

In [ ]:
pl = pa.sort_values("lits_nuits_total", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(range(len(pl)), 100 * pl.part_des_lits_nuits, color="C0")
ax.set_xticks(range(len(pl)), [nom(c) for c in pl.index], rotation=75, fontsize=8)
ax.set_ylabel("% des lits-nuits (2019-2022)")
ax2 = ax.twinx(); ax2.plot(range(len(pl)), 100 * pl.part_des_lits_nuits.cumsum(), color="C3", marker="o")
ax2.set_ylabel("% cumulé", color="C3"); ax2.set_ylim(0, 100); ax2.grid(False)
ax.set_title("Lits-nuits consommés par acte chirurgical (diagramme de Pareto)")
plt.show()
display(Markdown(f"Les 10 premiers actes représentent **{pl.part_des_lits_nuits.head(10).sum():.0%}** des lits-nuits, "
                 f"pour **{pl.effectif.head(10).sum() / len(df):.0%}** des interventions."))

In [ ]:
top_a = df.acte_chirurgical.value_counts().head(15).index
d = df[df.acte_chirurgical.isin(top_a)].copy()
d["acte"] = d.acte_chirurgical.map(nom)
ordre = d.groupby("acte").nuits_total.mean().sort_values().index
fig, axs = plt.subplots(1, 2, figsize=(17, 6.5))
sns.boxplot(data=d, y="acte", x="nuits_total", order=ordre, showfliers=False, ax=axs[0])
axs[0].set_title("Nuits selon l'acte chirurgical (15 plus fréquents)"); axs[0].set_ylabel("")
(d.groupby("acte").ambulatoire.mean().reindex(ordre) * 100).plot.barh(ax=axs[1]); axs[1].set_ylabel("")
axs[1].set_xlabel("% ambulatoire"); axs[1].set_title("Part d'ambulatoire")
plt.tight_layout(); plt.show()

### 4.4 Durée en salle par acte, et sa prévisibilité
Un acte à forte variabilité justifie une marge de sécurité plus grande dans la vacation.

In [ ]:
ordre = d.groupby("acte").duree_salle_min.median().sort_values().index
fig, axs = plt.subplots(1, 2, figsize=(17, 6.5))
sns.boxplot(data=d, y="acte", x="duree_salle_min", order=ordre, showfliers=False, ax=axs[0])
axs[0].set_title("Durée en salle (min)"); axs[0].set_ylabel("")
pp = pa[pa.effectif >= 50]
sns.scatterplot(data=pp, x="duree_salle_mediane", y="duree_salle_cv", size="effectif", sizes=(20, 400), hue="appareil", ax=axs[1])
for c, r in pp.sort_values("duree_salle_cv", ascending=False).head(6).iterrows():
    axs[1].annotate(nom(c)[:30], (r.duree_salle_mediane, r.duree_salle_cv), fontsize=7, xytext=(3, 3), textcoords="offset points")
axs[1].set_xlabel("durée médiane (min)"); axs[1].set_ylabel("coefficient de variation")
axs[1].set_title("Prévisibilité de la durée (en haut : actes imprévisibles)"); axs[1].legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

### 4.5 Sous arthroscopie ou à ciel ouvert

In [ ]:
art = df.groupby(df.acte_sous_arthroscopie.map({True: "arthroscopie (voie C)", False: "autres voies"})).agg(
    interventions=("id_sejour", "size"), part_ambulatoire=("ambulatoire", "mean"), nuits_moyenne=("nuits_total", "mean"),
    duree_salle_mediane=("duree_salle_min", "median"))
display(art.round(2))

### 4.6 Le virage ambulatoire, acte par acte
Évolution de la part d'ambulatoire entre 2019 et 2022 pour les actes où elle a le plus changé.

In [ ]:
ev = pa.dropna(subset=["part_ambulatoire_2019", "part_ambulatoire_2022"])
ev = ev[ev.effectif >= 50].assign(evolution=lambda x: x.part_ambulatoire_2022 - x.part_ambulatoire_2019)
ev = ev.reindex(ev.evolution.abs().sort_values(ascending=False).index).head(12)
fig, ax = plt.subplots(figsize=(10, 5))
for i, (c, r) in enumerate(ev.iterrows()):
    ax.plot([100 * r.part_ambulatoire_2019, 100 * r.part_ambulatoire_2022], [i, i], color="0.6")
    ax.scatter(100 * r.part_ambulatoire_2019, i, color="C0", zorder=3)
    ax.scatter(100 * r.part_ambulatoire_2022, i, color="C3", zorder=3)
ax.set_yticks(range(len(ev)), [nom(c) for c in ev.index], fontsize=8)
ax.set_xlabel("% ambulatoire (bleu : 2019, rouge : 2022)"); ax.set_title("Évolution de l'ambulatoire par acte")
plt.show()

### 4.7 Les actes « lourds » : quel jour sont-ils opérés ?
Les actes qui remplissent le plus les lits sont ceux dont le jour d'opération pèse sur le week-end. C'est le lien direct entre la grille de vacations et l'occupation des lits.

In [ ]:
lourds = pa.sort_values("lits_nuits_total", ascending=False).head(12).index
h = df[df.acte_chirurgical.isin(lourds)]
t = pd.crosstab(h.acte_chirurgical.map(nom), h.jour_semaine_nom, normalize="index") * 100
t = t[[j for j in t.columns if j in ("lundi", "mardi", "mercredi", "jeudi", "vendredi")]]
plt.figure(figsize=(10, 6))
sns.heatmap(t, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% des interventions de l'acte"})
plt.title("Jour d'opération des 12 actes qui consomment le plus de lits"); plt.ylabel(""); plt.xlabel("")
plt.show()
display(pa.loc[lourds, ["libelle_le_plus_frequent", "nuits_moyenne", "part_operes_jeudi_vendredi", "part_sejours_avec_weekend"]].round(2))

### 4.8 Qui fait quoi ? Actes et chirurgiens

In [ ]:
t = pd.crosstab(df[df.acte_chirurgical.isin(top_a)].acte_chirurgical.map(nom), df.chirurgien.astype(str))
t = t.loc[:, t.sum() > 20]
plt.figure(figsize=(15, 7))
sns.heatmap(t, annot=True, fmt="d", cmap="Greens", cbar=False)
plt.title("Nombre d'interventions par acte et par chirurgien (2019-2022)"); plt.ylabel(""); plt.xlabel("chirurgien")
plt.show()

### 4.9 Cohérence entre l'acte et le libellé
Un même acte peut porter plusieurs libellés, et un libellé recouvrir plusieurs actes : c'est ce qui rend le libellé seul imprécis.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(pa.part_de_ce_libelle * 100, bins=20, ax=axs[0])
axs[0].set_xlabel("% du libellé le plus fréquent pour l'acte"); axs[0].set_title("Un acte, combien de libellés ?")
lib = df.groupby("type_intervention_norm").acte_chirurgical.nunique()
lib = lib[df.type_intervention_norm.value_counts().head(15).index].sort_values()
lib.plot.barh(ax=axs[1]); axs[1].set_xlabel("actes CCAM distincts"); axs[1].set_title("Un libellé, combien d'actes ? (15 libellés les plus fréquents)")
plt.tight_layout(); plt.show()
display(Markdown("**Actes les moins cohérents avec un libellé unique** :"))
display(pa[pa.effectif >= 50].sort_values("part_de_ce_libelle").head(8)[["libelle_le_plus_frequent", "part_de_ce_libelle", "effectif"]])

### 4.10 Plusieurs actes dans la même intervention

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=df, x="nb_actes_chirurgicaux", y="duree_salle_min", ax=axs[0]); axs[0].set_title("Durée en salle selon le nombre d'actes")
sns.barplot(data=df, x="nb_actes_chirurgicaux", y="nuits_total", ax=axs[1]); axs[1].set_title("Nuits selon le nombre d'actes")
plt.tight_layout(); plt.show()

## 5. Ce qui détermine la durée d'hospitalisation : diagnostic, acte, âge

Le chirurgien référent a indiqué que **le diagnostic principal, l'acte et l'âge** sont les critères qui agissent le plus. Cette section le vérifie sur les données, avant de construire un modèle de prédiction (section 6). L'objectif reste le même : prévoir, dès la consultation, combien de nuits chaque patient occupera un lit, pour **lisser l'occupation des lits**.

### 5.1 Le diagnostic principal

In [ ]:
# libellé lisible d'une catégorie de diagnostic : motif clinique de la classification d'urgence, sinon libellé d'intervention le plus fréquent
def _libelle(g):
    if not g.regle_urgence.eq("défaut").all():
        return g.motif_urgence.mode().iloc[0]
    m = g.type_intervention_norm.dropna().mode()
    return m.iloc[0] if len(m) else ""
lib_diag = df.groupby("diag_categorie").apply(_libelle)
nom_diag = lambda c: f"{c} — {str(lib_diag.get(c, ''))[:34]}"
top_d = df.diag_categorie.value_counts().head(20).index
d = df[df.diag_categorie.isin(top_d)].copy(); d["diagnostic"] = d.diag_categorie.map(nom_diag)
ordre = d.groupby("diagnostic").nuits_total.mean().sort_values().index
fig, axs = plt.subplots(1, 2, figsize=(17, 7))
sns.boxplot(data=d, y="diagnostic", x="nuits_total", order=ordre, showfliers=False, ax=axs[0])
axs[0].set_title("Nuits selon le diagnostic principal (20 plus fréquents)"); axs[0].set_ylabel("")
(d.groupby("diagnostic").ambulatoire.mean().reindex(ordre) * 100).plot.barh(ax=axs[1]); axs[1].set_ylabel("")
axs[1].set_xlabel("% ambulatoire"); axs[1].set_title("Part d'ambulatoire")
plt.tight_layout(); plt.show()

### 5.2 L'âge

In [ ]:
df["tranche_age_5"] = (df.age // 5 * 5).clip(15, 90)
fig, axs = plt.subplots(1, 3, figsize=(18, 4.3))
sns.lineplot(data=df, x="tranche_age_5", y="nuits_total", errorbar=("ci", 95), ax=axs[0], marker="o")
axs[0].set_title("Nuits moyennes selon l'âge (IC 95 %)"); axs[0].set_xlabel("âge (tranches de 5 ans)")
sns.lineplot(data=df, x="tranche_age_5", y="ambulatoire", errorbar=None, ax=axs[1], marker="o")
axs[1].set_title("Part d'ambulatoire selon l'âge"); axs[1].set_xlabel("âge")
lourds = df.acte_chirurgical.value_counts().index[:40]
lourds = [a for a in pa.sort_values("lits_nuits_total", ascending=False).index if a in lourds][:5]
for a in lourds:
    g = df[df.acte_chirurgical == a].groupby(pd.cut(df.age, [0, 50, 60, 70, 80, 120]), observed=True).nuits_total.mean()
    axs[2].plot([str(i) for i in g.index], g.values, marker="o", label=nom(a)[:30])
axs[2].set_title("Nuits selon l'âge, pour les actes qui consomment le plus de lits"); axs[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

### 5.3 Acte et âge ensemble

In [ ]:
top12 = df.acte_chirurgical.value_counts().head(12).index
h = df[df.acte_chirurgical.isin(top12)]
t = h.pivot_table(index=h.acte_chirurgical.map(nom), columns="classe_age", values="nuits_total", aggfunc="mean", observed=False)
plt.figure(figsize=(10, 6)); sns.heatmap(t, annot=True, fmt=".1f", cmap="Reds")
plt.title("Nuits moyennes selon l'acte et l'âge"); plt.ylabel(""); plt.show()

### 5.4 Quel critère explique le plus ? Part de variance expliquée

Pour chaque critère, on prédit la durée d'un séjour de **2022** par la moyenne des séjours de **2019-2021** qui ont la même valeur (même acte, même tranche d'âge…), puis on mesure le **R²** : la part des variations de durée que ce critère explique (0 = rien, 1 = tout). Calculé sur des années différentes de celles qui servent à estimer les moyennes, ce R² ne peut pas être gonflé artificiellement par un critère à très nombreuses modalités. On le calcule pour la durée d'hospitalisation **et** pour la durée opératoire, puisque c'est sur cette dernière que portait la remarque du médecin.

In [ ]:
def r2_moyennes(cles, cible, tr, te):
    m = tr.groupby(cles, observed=True)[cible].mean()
    p = te.set_index(cles).index.map(m.to_dict()) if len(cles) > 1 else te[cles[0]].map(m)
    p = pd.Series(np.asarray(p, dtype=float), index=te.index).fillna(tr[cible].mean())
    y = te[cible]; ok = y.notna()
    return 1 - ((y[ok] - p[ok]) ** 2).sum() / ((y[ok] - y[ok].mean()) ** 2).sum()

tr = df[(df.annee < 2022) & ~df.periode_covid].copy(); te = df[df.annee == 2022].copy()
for x in (tr, te):
    x["age_5ans"] = (x.age // 5).astype(int); x["comorbidites"] = x[[c for c in df.columns if c.startswith("comorbidite_")]].sum(axis=1)
criteres = {"âge (tranches de 5 ans)": ["age_5ans"], "sexe": ["sexe"], "comorbidités (nombre)": ["comorbidites"],
            "chirurgien": ["chirurgien"], "classe d'urgence": ["classe_urgence"], "diagnostic (catégorie)": ["diag_categorie"],
            "diagnostic (code complet)": ["diag_principal"], "libellé d'intervention": ["type_intervention_norm"],
            "acte chirurgical": ["acte_chirurgical"], "acte + âge": ["acte_chirurgical", "classe_age"],
            "diagnostic + acte": ["diag_categorie", "acte_chirurgical"],
            "diagnostic + acte + âge": ["diag_categorie", "acte_chirurgical", "classe_age"]}
r2 = pd.DataFrame({c: {"durée d'hospitalisation (nuits)": r2_moyennes(k, "nuits_total", tr, te),
                       "durée opératoire (min)": r2_moyennes(k, "duree_salle_min", tr, te)} for c, k in criteres.items()}).T
display(r2.round(3))
ax = r2.plot.barh(figsize=(10, 6)); ax.set_xlabel("R² sur 2022 (part de variance expliquée)")
ax.set_title("Pouvoir explicatif de chaque critère"); plt.show()

**Lecture** : l'**acte** est le critère qui, seul, explique le plus la durée d'hospitalisation comme la durée opératoire ; le **diagnostic** apporte une information proche (un diagnostic appelle souvent un acte) ; l'**âge seul** explique peu, mais il **module** la durée d'un même acte (section 5.3). Combiner les critères n'aide que jusqu'à un certain point : avec trop de combinaisons, chaque case contient trop peu de patients pour estimer une moyenne fiable. C'est précisément ce que règlent les modèles de machine learning de la section suivante.

## 6. Prédire la durée d'hospitalisation d'un nouveau patient

**Protocole** (cours 2 et 3) :
- **apprentissage sur 2019-2021** (hors confinements), **test sur 2022** : on prédit l'avenir à partir du passé ;
- uniquement des variables **connues à la consultation** (diagnostic, acte prévu, âge, sexe, chirurgien, comorbidités, historique, classe d'urgence) ;
- comparaison à des **références simples** : un modèle qui ne les bat pas n'apporte rien.

### 6.1 Quelles variables, quel modèle ?
Erreur moyenne absolue (MAE, en nuits) sur 2022, pour plusieurs jeux de variables et plusieurs modèles du cours.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from prediction_sejour import VARIABLES_PRED

jeux = {"âge seul": ["age"], "diagnostic seul": ["diag_categorie", "diag_principal"], "acte seul": ["acte_chirurgical"],
        "diagnostic + acte + âge": ["diag_categorie", "diag_principal", "acte_chirurgical", "age"],
        "toutes les variables": VARIABLES_PRED}
modeles = {"régression linéaire": lambda: LinearRegression(),
           "arbre de décision": lambda: DecisionTreeRegressor(max_depth=8, min_samples_leaf=20, random_state=0),
           "forêt aléatoire": lambda: RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0),
           "gradient boosting": lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.06, random_state=0)}
base_ml = df[~df.periode_covid]
mae, pred_reg = {}, {}
for nj, vars_ in jeux.items():
    X, y = jeu_ml(base_ml, cible="nuits_total", variables=vars_, top=40)
    Xtr, Xte, ytr, yte = decoupage_temporel(base_ml, X, y)
    for nm, f in modeles.items():
        m = f().fit(Xtr, ytr); p = np.clip(m.predict(Xte), 0, None)
        mae[(nj, nm)] = mean_absolute_error(yte, p)
        if nj == "toutes les variables" and nm == "gradient boosting":
            modele_gb, X_test_gb, y_test_gb = m, Xte, yte
tr = base_ml[base_ml.annee < 2022]; te = base_ml[base_ml.annee == 2022]
ref_med = mean_absolute_error(te.nuits_total, te.acte_chirurgical.map(tr.groupby("acte_chirurgical").nuits_total.median()).fillna(tr.nuits_total.median()))
ref_moy = mean_absolute_error(te.nuits_total, te.acte_chirurgical.map(tr.groupby("acte_chirurgical").nuits_total.mean()).fillna(tr.nuits_total.mean()))
t = pd.Series(mae).unstack()[list(modeles)].loc[list(jeux)]
display(Markdown(f"**Références** : médiane de l'acte → MAE **{ref_med:.3f}** nuit ; moyenne de l'acte → **{ref_moy:.3f}** nuit."))
display(t.round(3))
plt.figure(figsize=(9, 4)); sns.heatmap(t, annot=True, fmt=".3f", cmap="RdYlGn_r"); plt.title("MAE (nuits) sur 2022 : plus bas = mieux"); plt.show()

**Lecture** :
- L'**âge seul** ne prédit presque rien ; le **diagnostic** et surtout la combinaison **diagnostic + acte + âge** portent l'information, et **ajouter les autres variables** (chirurgien, comorbidités, classe d'urgence…) améliore encore tous les modèles. Le **gradient boosting** et la **forêt aléatoire** sont les meilleurs ; l'arbre seul et la régression linéaire restent en retrait.
- **Mais aucun de ces modèles ne bat la simple médiane de l'acte** (0,41 nuit). Ce n'est pas un échec des modèles : entraînés à minimiser l'erreur au carré, ils prédisent une **moyenne**, alors que la MAE récompense la **médiane** (beaucoup de séjours à 0 nuit, quelques séjours très longs). C'est un piège classique : il faut choisir le modèle selon l'usage.
- Or l'usage, ici, n'est pas de deviner un nombre de nuits exact : c'est d'**additionner les patients nuit par nuit**. Il faut donc des **probabilités** (section 6.2) ; la médiane qu'on en tire bat d'ailleurs la médiane de l'acte.

### 6.2 Le modèle retenu : une probabilité pour chaque durée

Pour additionner les patients nuit par nuit, le modèle prédit pour chaque patient la **probabilité d'entrer la veille** et la **probabilité de rester 0, 1, 2… 7 nuits et plus après l'opération** (`PredicteurSejour`, gradient boosting). On en tire un **profil de lits** : les lits occupés en moyenne la veille, le soir de l'opération, le lendemain… Il est comparé au **profil historique de l'acte** (la distribution observée de 2019 à 2021 pour le même acte).

In [ ]:
from prediction_sejour import PredicteurSejour, ProfilHistorique, evaluer, log_loss_post, lits_attendus_par_nuit, lits_reels_par_nuit
pred_modele = PredicteurSejour().fit(tr)
pred_hist = ProfilHistorique().fit(tr)
res = {}
for nm, mo in (("profil historique de l'acte", pred_hist), ("modèle (gradient boosting)", pred_modele)):
    p = mo.predire(te)
    res[nm] = {**evaluer(p, te), "log-loss (qualité des probabilités, plus bas = mieux)": log_loss_post(mo, te)}
display(pd.DataFrame(res).round(3))
display(Markdown(f"Pour comparaison, la médiane de l'acte (section 6.1) fait une erreur de **{ref_med:.3f}** nuit : la valeur la plus probable "
                 "donnée par le modèle (« médiane prévue ») fait mieux, et le modèle fournit en plus les probabilités nécessaires au calcul des lits."))

# calibration : quand le modèle annonce 30 % d'ambulatoire, y en a-t-il vraiment 30 % ?
p = pred_modele.predire(te)
cal = pd.DataFrame({"prévu": p.p_ambulatoire, "réel": te.ambulatoire.astype(int)})
cal["tranche"] = pd.cut(cal.prévu, np.linspace(0, 1, 11))
c = cal.groupby("tranche", observed=True).agg(prévu=("prévu", "mean"), réel=("réel", "mean"), n=("réel", "size"))
p3 = 1 - p[[f"p_post_{k}" for k in range(3)]].sum(axis=1)
cal3 = pd.DataFrame({"prévu": p3, "réel": (te.nuits_apres_op >= 3).astype(int)})
cal3["tranche"] = pd.cut(cal3.prévu, np.linspace(0, 1, 11))
c3 = cal3.groupby("tranche", observed=True).agg(prévu=("prévu", "mean"), réel=("réel", "mean"))
fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
for ax, cc, titre in ((axs[0], c, "Probabilité d'ambulatoire"), (axs[1], c3, "Probabilité de rester 3 nuits ou plus")):
    ax.plot([0, 1], [0, 1], ls=":", c="k"); ax.plot(cc.prévu, cc.réel, marker="o")
    ax.set_xlabel("probabilité prévue"); ax.set_ylabel("fréquence observée en 2022"); ax.set_title(f"Calibration — {titre}")
plt.tight_layout(); plt.show()

### 6.3 Le test qui compte : les lits de 2022

On place chaque patient de 2022 à sa vraie date d'opération et on additionne les profils de lits prévus. Si les prévisions sont bonnes, la courbe des lits **prévus** doit suivre celle des lits **réellement occupés**. C'est exactement le calcul que fera l'outil de programmation pour décider de la date d'un nouveau patient.

In [ ]:
deb, fin_ = "2022-01-10", "2022-12-18"
reel22 = lits_reels_par_nuit(df, deb, fin_)
prev_m = lits_attendus_par_nuit(te, pred_modele.profil_lits(te), deb, fin_)
prev_h = lits_attendus_par_nuit(te, pred_hist.profil_lits(te), deb, fin_)
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(reel22.index, reel22.values, color="0.4", lw=1, label="lits réellement occupés")
ax.plot(prev_h.index, prev_h.values, color="C1", lw=1, label="prévus : profil historique de l'acte")
ax.plot(prev_m.index, prev_m.values, color="C0", lw=1.3, label="prévus : modèle")
ax.set_ylabel("lits"); ax.set_title("Lits occupés chaque nuit en 2022 : réalité et prévisions faites à la consultation"); ax.legend()
plt.show()
t = pd.DataFrame({nm: {"erreur moyenne (lits par nuit)": (s - reel22).abs().mean(), "biais moyen (lits)": (s - reel22).mean(),
                       "corrélation avec la réalité": s.corr(reel22), "erreur sur le pic hebdomadaire (lits)":
                       (s.resample("W").max() - reel22.resample("W").max()).abs().mean()}
                  for nm, s in (("profil historique de l'acte", prev_h), ("modèle", prev_m))})
display(t.round(3))
display(Markdown("**Lecture** : le modèle suit nettement mieux les lits réels que le profil historique de l'acte (erreur et pics plus faibles). "
                 "Les deux **surestiment** légèrement les lits de 2022 : les séjours raccourcissent d'une année sur l'autre (virage ambulatoire), "
                 "et un modèle appris sur 2019-2021 retarde un peu. En pratique, il faut **réentraîner le modèle régulièrement** "
                 "(par exemple chaque trimestre) sur les données les plus récentes."))
jours = pd.DataFrame({"réel": reel22, "modèle": prev_m, "historique": prev_h})
jours.groupby(jours.index.weekday).mean().rename(index=dict(enumerate(JOURS))).plot.bar(figsize=(9, 3.5), rot=0,
    title="Lits moyens selon la nuit (2022)"); plt.show()

### 6.4 Quelles variables comptent pour le modèle ?
Importance par permutation (méthode indépendante du modèle) : on mélange au hasard une variable dans les données de test et on mesure de combien l'erreur augmente. Les indicatrices d'une même variable (tous les actes, tous les diagnostics…) sont mélangées ensemble.

In [ ]:
rng = np.random.default_rng(0)
base_err = mean_absolute_error(y_test_gb, modele_gb.predict(X_test_gb))
groupes = {v: [c for c in X_test_gb.columns if c == v or c.startswith(v + "_")] for v in VARIABLES_PRED}
imp = {}
for v, cols in groupes.items():
    if not cols:
        continue
    errs = []
    for _ in range(3):
        Xp = X_test_gb.copy(); perm = rng.permutation(len(Xp))
        Xp[cols] = Xp[cols].values[perm]
        errs.append(mean_absolute_error(y_test_gb, modele_gb.predict(Xp)))
    imp[v] = np.mean(errs) - base_err
imp = pd.Series(imp).sort_values(ascending=False)
plt.figure(figsize=(9, 6)); sns.barplot(x=imp.values, y=imp.index, color="C0")
plt.xlabel("hausse de l'erreur (nuits) quand la variable est brouillée"); plt.title("Importance des variables (durée d'hospitalisation)")
plt.show()

### 6.5 La durée opératoire
Même démarche pour la durée en salle (utile pour remplir les vacations) : prédiction de la durée **et** d'une durée « haute » (90e centile) qui sert de marge de sécurité, comparée à la médiane et au 90e centile de l'acte.

In [ ]:
var_op = ["diag_categorie", "diag_principal", "acte_chirurgical", "age", "chirurgien", "nb_actes_chirurgicaux",
          "type_intervention_norm", "sexe", "nb_diag_associes"]
base_op = base_ml[base_ml.duree_salle_min.notna()]
X, y = jeu_ml(base_op, cible="duree_salle_min", variables=var_op, top=40)
Xtr, Xte, ytr, yte = decoupage_temporel(base_op, X, y)
tro, teo = base_op.loc[Xtr.index], base_op.loc[Xte.index]
m50 = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, random_state=0).fit(Xtr, ytr)
m90 = HistGradientBoostingRegressor(loss="quantile", quantile=0.9, max_iter=300, random_state=0).fit(Xtr, ytr)
med_acte = teo.acte_chirurgical.map(tro.groupby("acte_chirurgical").duree_salle_min.median()).fillna(tro.duree_salle_min.median())
p90_acte = teo.acte_chirurgical.map(tro.groupby("acte_chirurgical").duree_salle_min.quantile(0.9)).fillna(tro.duree_salle_min.quantile(0.9))
p50, p90 = m50.predict(Xte), m90.predict(Xte)
display(pd.DataFrame({
    "médiane / 90e centile de l'acte": {"MAE (min)": mean_absolute_error(yte, med_acte), "dépassent la durée haute (%)": 100 * (yte > p90_acte).mean(),
                                        "marge moyenne prévue (min)": (p90_acte - med_acte).mean()},
    "modèle (diagnostic, acte, âge, chirurgien…)": {"MAE (min)": mean_absolute_error(yte, p50), "dépassent la durée haute (%)": 100 * (yte > p90).mean(),
                                                    "marge moyenne prévue (min)": (p90 - p50).mean()}}).round(2))

Une durée haute bien calibrée doit être dépassée par **environ 10 %** des interventions ; ici 12 à 13 %, la durée haute est donc un peu optimiste (viser le 92e centile corrigerait cet écart). Le modèle prédit mieux la durée que la médiane de l'acte et donne une marge **propre à chaque patient**, plus petite en moyenne, qui remplace avantageusement la marge P90 − médiane additionnée patient par patient du modèle de planning.

### 6.6 Le modèle final, prêt pour la programmation
Le modèle est réentraîné sur **2019-2022** (hors confinements) et enregistré dans `modeles/predicteur_sejour.joblib`. Exemple pour trois nouveaux patients vus en consultation :

In [ ]:
final = PredicteurSejour().fit(base_ml)
final.sauver("modeles/predicteur_sejour.joblib")
exemples = pd.DataFrame([
    {"patient": "prothèse totale de genou, 72 ans, hypertendue", "age": 72, "sexe": "F", "diag_principal": "M17.1", "acte_chirurgical": "NFKA007",
     "chirurgien": df[df.acte_chirurgical == "NFKA007"].chirurgien.mode()[0], "type_intervention_norm": "prothese totale genou", "comorbidite_hypertension": True},
    {"patient": "arthroscopie du genou, 35 ans", "age": 35, "sexe": "H", "diag_principal": "M23.22", "acte_chirurgical": "NFFC004",
     "chirurgien": df[df.acte_chirurgical == "NFFC004"].chirurgien.mode()[0], "type_intervention_norm": "arthroscopie genou"},
    {"patient": "fracture du col du fémur, 84 ans, anticoagulée", "age": 84, "sexe": "F", "diag_principal": "S72.00", "acte_chirurgical": "NEKA012",
     "chirurgien": df[df.acte_chirurgical == "NEKA012"].chirurgien.mode()[0], "type_intervention_norm": "prothese totale hanche", "comorbidite_anticoagulant": True},
]).set_index("patient")
from urgences import classer_df
exemples["diag_categorie"] = exemples.diag_principal.str[:3]
exemples["acte_topographie"] = exemples.acte_chirurgical.str[:2]
for c in ["comorbidite_hypertension", "comorbidite_diabete", "comorbidite_obesite", "comorbidite_tabac", "comorbidite_anticoagulant"]:
    exemples[c] = exemples.get(c, pd.Series(False, index=exemples.index)).fillna(False).astype(bool)
exemples = pd.concat([exemples, classer_df(exemples)[["classe_urgence", "delai_max_jours", "score_priorite"]]], axis=1)
pe = final.predire(exemples)
display(pd.concat([exemples[["classe_urgence", "delai_max_jours"]], pe[["p_ambulatoire", "p_veille", "nuits_esperees",
                   "nuits_apres_op_mediane", "nuits_apres_op_p90"]]], axis=1).round(2))
prof = final.profil_lits(exemples, horizon=10)
fig, ax = plt.subplots(figsize=(10, 3.8))
for i, nm_ in enumerate(exemples.index):
    ax.plot(range(-1, 10), prof[i], marker="o", label=nm_)
ax.set_xlabel("nuit par rapport à l'opération (−1 = veille)"); ax.set_ylabel("lit occupé (probabilité)")
ax.set_title("Profil de lits prévu à la consultation"); ax.legend(fontsize=8); plt.show()

**Comment l'utiliser pour programmer** : pour un nouveau patient, le profil ci-dessus est ajouté, à chaque date candidate, aux lits déjà prévus ; on choisit la date qui garde l'occupation la plus proche de la cible (moins de lits le week-end et en vacances), **dans le délai imposé par sa classe d'urgence** (section 7).

## 7. Classification des urgences

Pour savoir **quels patients opérer en premier**, chaque diagnostic principal reçoit une **classe d'urgence** avec un délai maximal. La classification n'est pas inventée : elle reprend trois références reconnues.

| Référence | Ce qu'elle apporte |
|---|---|
| **FSSA** — *Clinical Guide to Surgical Prioritisation*, Federation of Surgical Specialty Associations et British Orthopaedic Association (Royaume-Uni, 2020) | 5 niveaux de délai (P1a < 24 h, P1b < 72 h, P2 < 1 mois, P3 < 3 mois, P4 > 3 mois) et la **liste des situations d'orthopédie** relevant de chacun |
| **NCEPOD** — *Classification of Intervention* (2004) | 4 niveaux d'urgence d'une intervention : immédiate, urgente, accélérée, programmée |
| **SFAR / SOFCOT / SFGG / SFPC** — recommandation R4.1 (2017) | fracture de l'extrémité supérieure du fémur : **opérer dans les 48 h** suivant l'admission pour réduire la mortalité |

Le module `urgences.py` applique ces références aux codes CIM-10 de la base : la règle au préfixe le plus précis l'emporte (S72.0 avant S72), et une fracture **ouverte** (5e caractère « 1 ») est toujours U1.

In [ ]:
from urgences import tableau_classes, tableau_regles, ordre_de_passage, CLASSES
display(tableau_classes())
display(Markdown(f"**{len(tableau_regles())} règles** (extrait) :"))
display(tableau_regles().head(40))

### 7.1 Répartition dans la base

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
df.classe_urgence.value_counts().sort_index().plot.bar(ax=ax, rot=0, color=["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c", "#1f77b4"])
ax.set_title("Interventions par classe d'urgence (2019-2022)"); plt.show()
for k in ["U1", "U2", "P2", "P3"]:
    g = df[df.classe_urgence == k].groupby("motif_urgence").size().sort_values(ascending=False).head(8)
    display(Markdown(f"**{k} — {CLASSES[k]['libelle']}** ({(df.classe_urgence == k).sum()} interventions) :")); display(g.to_frame("interventions"))
display(Markdown(f"Diagnostics classés **par défaut** (sans règle précise) : **{(df.regle_urgence == 'défaut').mean():.1%}** des interventions, "
                 f"principalement : {', '.join(df[df.regle_urgence == 'défaut'].diag_categorie.value_counts().head(6).index)}."))

### 7.2 Les classes se vérifient-elles dans les données ?
La classification n'a utilisé que le diagnostic. Si elle est juste, les patients classés urgents doivent **se comporter comme des urgences** dans les données : plus souvent opérés le week-end, plus souvent hospitalisés avant l'opération (admis puis opérés le lendemain), répartis sur toute l'année.

In [ ]:
v = df.groupby("classe_urgence", observed=True).agg(
    interventions=("id_sejour", "size"), opérés_le_week_end_pct=("jour_semaine", lambda s: 100 * (s >= 5).mean()),
    hospitalisés_avant_op_pct=("nuits_avant_op", lambda s: 100 * (s > 0).mean()), nuits_moyennes=("nuits_total", "mean"),
    ambulatoire_pct=("ambulatoire", lambda s: 100 * s.mean()), âge_moyen=("age", "mean"),
    opérés_en_vacances_pct=("vacances_scolaires", lambda s: 100 * s.mean()))
display(v.round(1))
fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, col, t in zip(axs, ["opérés_le_week_end_pct", "hospitalisés_avant_op_pct", "opérés_en_vacances_pct"],
                      ["% opérés le week-end", "% hospitalisés avant l'opération", "% opérés pendant les vacances"]):
    v[col].plot.bar(ax=ax, rot=0, color=["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c", "#1f77b4"]); ax.set_title(t)
plt.tight_layout(); plt.show()

### 7.3 Combien d'urgences, et quand ? (dimensionner les plages URGENCES et la réserve de lits)

In [ ]:
u = df[df.urgence & ~df.periode_covid]
sem = u.set_index("date_inter").resample("W-MON").size()
fig, axs = plt.subplots(1, 3, figsize=(17, 3.8))
sns.histplot(sem, discrete=True, ax=axs[0]); axs[0].set_title("Urgences (U1 + U2) par semaine"); axs[0].set_xlabel("urgences dans la semaine")
u.groupby("jour_semaine_nom", observed=False).size().plot.bar(ax=axs[1], rot=30); axs[1].set_title("Selon le jour")
u.groupby("mois").size().div(df[~df.periode_covid].date_inter.dt.to_period("Y").nunique()).plot.bar(ax=axs[2], rot=0)
axs[2].set_title("Par mois (moyenne annuelle)")
plt.tight_layout(); plt.show()
occ_u = occupation_journaliere(u.assign())
display(Markdown(f"- **{sem.mean():.1f} urgences par semaine** en moyenne (90 % des semaines : moins de {sem.quantile(0.9):.0f}).\n"
                 f"- Les urgences occupent en moyenne **{occ_u.lits_occupes_nuit.mean():.1f} lits** chaque nuit "
                 f"(90 % des nuits : moins de {occ_u.lits_occupes_nuit.quantile(0.9):.0f}) : c'est l'ordre de grandeur de la **réserve de lits** à prévoir."))

### 7.4 L'ordre de passage
À l'intérieur d'une même classe, la FSSA recommande de départager selon le risque et le temps déjà attendu. Le **score de priorité** combine la classe (qui domine toujours), l'âge, les comorbidités, puis **+1 point par semaine d'attente** au moment de programmer. Les patients qui ont dépassé leur délai maximal passent en tête. Exemple sur une file d'attente fictive tirée de la base :

In [ ]:
file = df.sample(12, random_state=3)[["id_sejour", "age", "diag_principal", "motif_urgence", "classe_urgence", "delai_max_jours", "score_priorite"]].copy()
aujourd_hui = pd.Timestamp("2023-01-16")
file["date_demande"] = aujourd_hui - pd.to_timedelta(np.random.default_rng(1).integers(0, 120, len(file)), unit="D")
display(ordre_de_passage(file, aujourd_hui)[["id_sejour", "age", "motif_urgence", "classe_urgence", "attente_jours",
                                              "score_avec_attente", "date_limite", "en_retard"]])

### 7.5 Limites — à valider avec le chirurgien
- Le **diagnostic seul** ne dit pas tout : la FSSA demande de **réévaluer** chaque patient (déficit neurologique, douleur, aggravation, état général). Un canal carpien (P4) devient P2 en cas d'amyotrophie, une hernie discale (P3) devient U1 en cas de syndrome de la queue de cheval. La classe calculée est une **classe par défaut**, que le chirurgien peut toujours surclasser dans l'outil.
- Certains codes sont ambigus : les « syndromes des loges » de la base sont souvent opérés en ambulatoire, ce qui évoque des **formes chroniques d'effort** plutôt que l'urgence aiguë que désigne le code T79.6.
- Les règles et délais sont dans `urgences.py` (liste `REGLES`) : faciles à relire et à modifier avec le chirurgien.

## 8. Les patients

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(data=df, x="age", hue="ambulatoire", bins=40, element="step", kde=True, ax=axs[0])
axs[0].set_title("Âge selon l'ambulatoire")
sns.barplot(data=df, x="classe_age", y="nuits_total", hue="sexe", ax=axs[1]); axs[1].set_title("Nuits moyennes par âge et sexe")
com = [c for c in df.columns if c.startswith("comorbidite_")]
r = pd.DataFrame({c.replace("comorbidite_", ""): [df[c].mean() * 100, df.loc[df[c], "nuits_total"].mean(),
                                                  df.loc[~df[c], "nuits_total"].mean()] for c in com},
                 index=["% patients", "nuits si présente", "nuits si absente"]).T
r[["nuits si présente", "nuits si absente"]].plot.bar(ax=axs[2], rot=20); axs[2].set_title("Comorbidités et nuits")
plt.tight_layout(); plt.show()
display(r.round(2))
display(Markdown(f"**{(df.nb_sejours_anterieurs > 0).mean():.0%}** des interventions concernent un patient déjà opéré dans la base "
                 f"(ex. second genou)."))

## 9. Le bloc opératoire

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(df.duree_salle_min, bins=50, ax=axs[0]); axs[0].set_title("Durée en salle (TROS, min)")
sns.histplot(df.heure_entree_salle_min / 60, bins=40, ax=axs[1]); axs[1].set_title("Heure d'entrée en salle")
axs[1].set_xlabel("heure")
sns.scatterplot(data=df.sample(3000, random_state=0), x="duree_salle_min", y="nuits_total", hue="ambulatoire",
                alpha=0.4, ax=axs[2]); axs[2].set_ylim(0, 15); axs[2].set_title("Durée en salle et nuits")
plt.tight_layout(); plt.show()
d = df[df.type_intervention_norm.isin(top)]
ordre = d.groupby("type_intervention_norm").duree_salle_min.median().sort_values().index
plt.figure(figsize=(10, 6))
sns.boxplot(data=d, y="type_intervention_norm", x="duree_salle_min", order=ordre, showfliers=False)
plt.title("Durée en salle selon le type d'intervention"); plt.ylabel(""); plt.show()
dec = df[["duree_avant_incision_min", "duree_chirurgie_min", "attente_sspi_pre_min"]].describe().T.round(1)
display(Markdown("**Décomposition du temps de bloc** (minutes) :")); display(dec)

## 10. L'occupation des lits

Lits occupés chaque nuit = séjours tels que *date d'entrée ≤ nuit < date de sortie* (calculé à partir des dates).

In [ ]:
occ = occupation_journaliere(df)
occ = occ[(occ.date >= "2019-01-07") & (occ.date <= "2022-12-25")]
fig, ax = plt.subplots(figsize=(14, 4))
for a, b in COVID:
    ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color="red", alpha=0.12)
ax.plot(occ.date, occ.lits_occupes_nuit, lw=0.6, color="0.5")
ax.plot(occ.date, occ.lits_occupes_nuit.rolling(28, center=True).mean(), color="k", lw=2, label="moyenne sur 4 semaines")
ax.set_ylabel("lits occupés"); ax.set_title("Lits occupés chaque nuit (rouge : confinements)"); ax.legend()
plt.show()
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
sns.boxplot(data=occ, x="jour_semaine_nom", y="lits_occupes_nuit", ax=axs[0]); axs[0].tick_params(axis="x", rotation=30)
axs[0].set_title("Lits occupés selon la nuit")
sns.boxplot(data=occ[occ.date.dt.weekday < 4], x="vacances_ou_ferie", y="lits_occupes_nuit", ax=axs[1])
axs[1].set_title("Nuits de semaine : hors vacances / vacances et fériés")
m = occ.groupby("jour_semaine_nom", observed=True)[["entrees", "sorties"]].mean()
m.plot.bar(ax=axs[2], rot=30); axs[2].set_title("Entrées et sorties moyennes par jour")
plt.tight_layout(); plt.show()
o = occ[~occ.periode_covid]
display(Markdown(f"Hors confinements : **{o.lits_occupes_nuit.mean():.1f} lits** en moyenne, écart-type **{o.lits_occupes_nuit.std():.1f}**, "
                 f"pic **{o.lits_occupes_nuit.max()}** ; nuits du vendredi au dimanche : **{o[o.nuit_weekend].lits_occupes_nuit.mean():.1f}**, "
                 f"autres nuits : **{o[~o.nuit_weekend].lits_occupes_nuit.mean():.1f}**."))

## 11. Profils des interventions et des chirurgiens
Ces tables (aussi dans `base_preparee/`, avec le profil des actes de la section 4) résument ce que le recuit simulé utilise : demande, durée de séjour, part d'ambulatoire, durée en salle.

In [ ]:
display(Markdown("**Interventions** (au moins 20 cas) :"))
display(profil_interventions(df).head(25))
pc = profil_chirurgiens(df)
display(Markdown("**Chirurgiens en 2022** :"))
display(pc[pc.annee == 2022].sort_values("interventions", ascending=False))
p22 = pc[pc.annee == 2022]
plt.figure(figsize=(8, 5))
sns.scatterplot(data=p22, x="part_ambulatoire", y="nuits_moyenne_hospitalises", size="interventions", sizes=(30, 600),
                hue="part_operes_jeudi_vendredi", palette="viridis")
for _, r in p22.iterrows():
    plt.annotate(r.chirurgien, (r.part_ambulatoire, r.nuits_moyenne_hospitalises), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.title("Profil des chirurgiens en 2022 (taille : activité ; couleur : part opérée jeudi/vendredi)")
plt.show()

## 12. Corrélations entre variables numériques

In [ ]:
num = ["age", "nb_diag_associes", "nb_actes_ccam", "nb_sejours_anterieurs", "duree_salle_min", "duree_chirurgie_min",
       "nuits_avant_op", "nuits_apres_op", "nuits_total", "jour_semaine"]
plt.figure(figsize=(9, 7))
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Corrélations"); plt.show()

## 13. Une base prête pour le machine learning

### 13.1 Quand chaque variable est-elle connue ?
Pour prédire en **consultation** la durée de séjour ou l'ambulatoire, on ne peut utiliser que ce qui est connu à ce moment. Le GHM par exemple contient la lettre « J » pour les séjours ambulatoires : il est calculé **après** la sortie, l'utiliser serait tricher.

In [ ]:
dico = dictionnaire(df)
display(dico.groupby("connue_a").variable.apply(lambda s: ", ".join(s)).to_frame("variables"))
display(dico)

### 13.2 Cibles possibles et matrices prêtes pour scikit-learn
`jeu_ml` applique le traitement vu en cours : imputation par la médiane, regroupement des modalités rares, encodage binaire (`get_dummies`). Il **refuse** les variables connues trop tard. `decoupage_temporel` apprend sur 2019-2021 et teste sur 2022 : on prédit l'avenir à partir du passé, ce qui est plus honnête qu'un `train_test_split` au hasard.

In [ ]:
display(pd.Series(CIBLES).to_frame("cible"))
X, y = jeu_ml(df, cible="ambulatoire")
X_train, X_test, y_train, y_test = decoupage_temporel(df, X, y, annee_test=2022)
print(f"X : {X.shape[0]} lignes x {X.shape[1]} colonnes, {int(X.isna().sum().sum())} valeur manquante")
print(f"apprentissage (2019-2021) : {len(X_train)} ; test (2022) : {len(X_test)}")
X.head()

### 13.3 Premier contrôle : la base contient-elle de l'information ?
Un premier modèle simple (les méthodes du cours), comparé à une **règle de référence** : prédire, pour chaque type d'intervention, ce qui était le plus fréquent dans le passé. Un modèle qui ne bat pas cette règle n'apporte rien. Ce n'est pas encore l'étude de machine learning, seulement une vérification que la base est exploitable.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error

tr = df.loc[X_train.index]; te = df.loc[X_test.index]
regle = tr.groupby("type_intervention_norm").ambulatoire.mean()
p_ref = te.type_intervention_norm.map(regle).fillna(tr.ambulatoire.mean())
res = [{"modèle": "règle : type d'intervention", "accuracy": accuracy_score(y_test, p_ref > 0.5), "AUC": roc_auc_score(y_test, p_ref)}]
for nom, m in (("arbre de décision", DecisionTreeClassifier(max_depth=8, random_state=0)),
               ("forêt aléatoire", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1))):
    m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    res.append({"modèle": nom, "accuracy": accuracy_score(y_test, p > 0.5), "AUC": roc_auc_score(y_test, p)})
display(Markdown("**Ambulatoire ou non (test 2022)**")); display(pd.DataFrame(res).round(3))

Xn, yn = jeu_ml(df, cible="nuits_total")
Xn_tr, Xn_te, yn_tr, yn_te = decoupage_temporel(df, Xn, yn)
regle = tr.groupby("type_intervention_norm").nuits_total.median()
res = [{"modèle": "règle : médiane du type", "MAE (nuits)": mean_absolute_error(yn_te, te.type_intervention_norm.map(regle).fillna(tr.nuits_total.median()))}]
for nom, m in (("arbre de décision", DecisionTreeRegressor(max_depth=8, random_state=0)),
               ("forêt aléatoire", RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1))):
    m.fit(Xn_tr, yn_tr)
    res.append({"modèle": nom, "MAE (nuits)": mean_absolute_error(yn_te, m.predict(Xn_te))})
display(Markdown("**Nombre de nuits (test 2022)**")); display(pd.DataFrame(res).round(3))
imp = pd.Series(m.feature_importances_, index=Xn.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 5)); sns.barplot(x=imp.values, y=imp.index, color="C0"); plt.title("Variables les plus utiles (forêt, nuits)"); plt.show()

## 14. Utiliser la base dans les autres codes

```python
from preparation_donnees import charger_base_preparee, jeu_ml, decoupage_temporel, vers_format_projet

df = charger_base_preparee()                     # recharge base_preparee/base_propre.pkl en ~1 s
X, y = jeu_ml(df, cible="nuits_total")           # ou "ambulatoire", "classe_sejour", "duree_salle_min"…
X_train, X_test, y_train, y_test = decoupage_temporel(df, X, y, annee_test=2022)

base_recuit = vers_format_projet(df)             # même format que donnees_reelles.charger_base()
```

Options utiles de `jeu_ml` : `avec_anesthesie=True` (prédiction après la consultation d'anesthésie), `avec_date=True` (quand la date est choisie : jour de la semaine, vacances…), `exclure_covid=True` (retirer les confinements), `variables=[...]` (choisir ses variables, les fuites sont refusées).

Fichiers de `base_preparee/` : `base_propre.pkl` / `.csv` / `base_preparee.xlsx` (base complète), `dictionnaire_variables.csv`, `rapport_qualite.csv`, `occupation_journaliere.csv` (lits, entrées, sorties par jour), `profil_interventions.csv`, `profil_actes.csv`, `profil_chirurgiens.csv`.